# Tome C4, Chapter 1: Measuring Information (Python companion)

Exercises first, solutions at the end. We compute entropies, check the chain rule and the data processing inequality on random laws, watch the exclusive-or paradox, test Pinsker's inequality and Chernoff's exponent, verify Shannon's grouping axiom (and see Rényi's entropies fail it), and count triangles and points with Shearer's lemma. Every number quoted in the chapter is recomputed here.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools, math, random
from fractions import Fraction
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(1)
random.seed(1)

def H(p):
    p = np.asarray(p, dtype=float).ravel()
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

def h(t):
    return H([t, 1 - t])

def D(p, q):
    p, q = np.asarray(p, float).ravel(), np.asarray(q, float).ravel()
    m = p > 0
    if np.any(q[m] == 0):
        return math.inf
    return float((p[m] * np.log2(p[m] / q[m])).sum())

## Exercises

**P1.1 (★) The numbers of Section 1.1.** Compute $h(0.01)$, $h(0.1)$, $h(1/3)$, and the entropy of a variable that is $0$ with probability $0.99$ and otherwise uniform on $\{1,\dots,1000\}$; compute also the entropy given the news "$X\neq0$". Plot $h$.

**P1.2 (★) Dice.** For two fair dice, compute exactly $H(S)$, $H(D)$, $H(S,D)$ and $I(S;D)$ for the sum $S$ and the absolute difference $D$ (Exercise 1.1).

**P1.3 (★★) Chain rule and data processing.** Draw random joint laws of $(X,Y,Z)$ on $3\times4\times3$ points. Check numerically $H(X,Y)=H(X)+H(Y\mid X)$, $I(X;Y)\ge0$, $I(X;Y\mid Z)\ge 0$, and, for Markov chains $X\to Y\to Z$ built from random channels, $I(X;Z)\le I(X;Y)$ (Theorem 1.5). Then reproduce the exclusive-or example of Exercise 1.3.

**P1.4 (★★) Pinsker and Chernoff.** (a) For random pairs of laws on $5$ points, compute the ratio $\delta(p,q)/\sqrt{\frac{\ln 2}{2}D(p\|q)}$ and check that it never exceeds $1$ (Theorem 1.8); find pairs of coins where it is close to $1$. (b) For $n$ fair coins and the event "at least $(\frac12+\varepsilon)n$ heads", compare $\frac1n\log_2\frac1{p(S)}$ with $d=d(\frac12+\varepsilon\|\frac12)$ and $2d+\frac2n$ (Exercise 1.6).

**P1.5 (★★) Fano.** A uniform $X\in\{0,\dots,7\}$ is observed through a channel that keeps it with probability $1-\alpha$ and replaces it by a uniform other value with probability $\alpha$. For the best guess, compute $P_e$ and $H(X\mid Y)$, and compare with the bound $h(P_e)+P_e\log_2 7$ of Theorem 1.9.

**P1.6 (★★) Shannon's axioms.** Check the grouping identity (iii) of Theorem 1.10 for Shannon entropy on random laws and random groupings, and show that the collision entropy $-\log_2\sum p_i^2$ and the min-entropy fail it; reproduce the numbers $1.585$ and $1.515$ of Section 1.6. Check also $H_\infty\le H_2\le H\le$ and $H_2\le 2H_\infty$ (Exercise 1.12), and the example of a law with one atom of mass $\frac12$.

**P1.7 (★★) Counting with entropy.** (a) Compare the number of triangles of complete graphs $K_k$ and of random graphs with the bound $\frac{\sqrt2}{3}m^{3/2}$ of Corollary 1.13. (b) Check Loomis–Whitney (Corollary 1.12) on random sets of points of $\{0,\dots,9\}^3$ and on a box. (c) Check the binomial bounds of Proposition 1.14 for $n=100$, and compute $\log_2 10!$ and $\log_2 1000!$ (Exercise 1.8).

## Solutions

### P1.1

In [ ]:
print("h(0.01) =", round(h(0.01), 4), "  h(0.1) =", round(h(0.1), 4), "  h(1/3) =", round(h(1/3), 4))
p = np.array([0.99] + [0.01 / 1000] * 1000)
print("H(X) =", round(H(p), 4), "  H(X | X != 0) =", round(math.log2(1000), 4))
t = np.linspace(0.0005, 0.9995, 400)
plt.figure(figsize=(5, 3)); plt.plot(t, [h(x) for x in t]); plt.xlabel("t"); plt.ylabel("h(t)"); plt.title("binary entropy"); plt.show()

A nearly certain event still carries $0.081$ bits, and the bad news $X\neq0$ raises the uncertainty from $0.18$ to $9.97$ bits: conditioning reduces entropy only on average.

### P1.2

In [ ]:
from collections import Counter
outs = [(a, b) for a in range(1, 7) for b in range(1, 7)]
S = Counter(a + b for a, b in outs); Dd = Counter(abs(a - b) for a, b in outs); SD = Counter((a + b, abs(a - b)) for a, b in outs)
HS, HD, HSD = (H(np.array(list(c.values())) / 36) for c in (S, Dd, SD))
print("H(S) = %.3f  H(D) = %.3f  H(S,D) = %.3f  I(S;D) = %.3f" % (HS, HD, HSD, HS + HD - HSD))
print("values of (S,D):", len(SD), " probabilities:", sorted(Counter(SD.values()).items()))

### P1.3

In [ ]:
def marg(P, axes):
    return P.sum(axis=tuple(a for a in range(P.ndim) if a not in axes))

def Hc(P, A, B):  # H(X_A | X_B) for a joint array P
    return H(marg(P, A | B)) - H(marg(P, B))

def I(P, A, B, Cs=frozenset()):
    return Hc(P, A, Cs) - Hc(P, A, B | Cs)

worst = 0
for _ in range(2000):
    P = rng.random((3, 4, 3)) ** 3; P /= P.sum()
    X, Y, Z = {0}, {1}, {2}
    worst = max(worst, abs(H(marg(P, X | Y)) - H(marg(P, X)) - Hc(P, Y, X)))
    assert I(P, X, Y) >= -1e-12 and I(P, X, Y, frozenset(Z)) >= -1e-12
print("max chain-rule error:", worst)

gap = []
for _ in range(2000):   # Markov chains X -> Y -> Z
    px = rng.dirichlet(np.ones(3)); K1 = rng.dirichlet(np.ones(4), size=3); K2 = rng.dirichlet(np.ones(3), size=4)
    P = px[:, None, None] * K1[:, :, None] * K2[None, :, :]
    gap.append(I(P, {0}, {1}) - I(P, {0}, {2}))
print("min of I(X;Y) - I(X;Z) over Markov chains:", min(gap))

P = np.zeros((2, 2, 2))
for x in (0, 1):
    for y in (0, 1):
        P[x, y, x ^ y] = 0.25
print("XOR: I(X;Y) =", I(P, {0}, {1}), " I(X;Y|Z) =", I(P, {0}, {1}, frozenset({2})), " I(X;Z) =", I(P, {0}, {2}), " I(X;Y,Z) =", I(P, {0}, {1, 2}))

### P1.4

In [ ]:
def tv(p, q):
    return 0.5 * np.abs(np.asarray(p) - np.asarray(q)).sum()

ratios = []
for _ in range(20000):
    p, q = rng.dirichlet(np.ones(5)), rng.dirichlet(np.ones(5))
    ratios.append(tv(p, q) / math.sqrt(math.log(2) / 2 * D(p, q)))
print("largest ratio, random laws on 5 points: %.4f" % max(ratios))
for eps in (0.1, 0.01, 0.001):
    p, q = [0.5 + eps, 0.5 - eps], [0.5, 0.5]
    print("coins 1/2+%g vs 1/2: ratio %.6f" % (eps, tv(p, q) / math.sqrt(math.log(2) / 2 * D(p, q))))

def dcoin(a, b):
    return D([a, 1 - a], [b, 1 - b])

eps = 0.1
print("\n   n   (1/n)log2(1/p(S))      d     2d+2/n")
for n in (50, 100, 200, 500, 1000):
    k = round((0.5 + eps) * n)
    pS = sum(math.comb(n, j) for j in range(k, n + 1)) / 2 ** n
    d = dcoin(0.5 + eps, 0.5)
    print("%4d   %10.5f        %.5f   %.5f" % (n, math.log2(1 / pS) / n, d, 2 * d + 2 / n))

The exact exponent tends to $d$ from above, as Chernoff's bound and the lower bound of the exercise predict, and Pinsker's ratio tends to $1$ for nearly fair coins: the constant is sharp.

### P1.5

In [ ]:
for alpha in (0.05, 0.2, 0.5, 0.8):
    K = np.full((8, 8), alpha / 7); np.fill_diagonal(K, 1 - alpha)
    P = K / 8                       # joint law of (X, Y)
    guess = P.argmax(axis=0)        # best guess of X for each y
    Pe = 1 - sum(P[guess[y], y] for y in range(8))
    HXY = H(P) - H(P.sum(axis=0))
    print("alpha=%.2f  Pe=%.3f  H(X|Y)=%.4f  Fano bound=%.4f" % (alpha, Pe, HXY, h(Pe) + Pe * math.log2(7)))

For $\alpha<\frac78$ the best guess is $Y$ itself, $P_e=\alpha$, and Fano's bound is an equality: given an error, $X$ is uniform on the $7$ other values.

### P1.6

In [ ]:
def H2(p):
    p = np.asarray(p, float); return -math.log2((p ** 2).sum())

def Hinf(p):
    return -math.log2(max(p))

def grouping_defect(F, p, blocks):
    q = [sum(p[i] for i in B) for B in blocks]
    return F(p) - F(q) - sum(qj * F([p[i] / qj for i in B]) for qj, B in zip(q, blocks))

worst = {"Shannon": 0, "collision": 0, "min": 0}
for _ in range(2000):
    n = rng.integers(2, 7); p = rng.dirichlet(np.ones(n))
    perm = rng.permutation(n); cut = sorted(rng.choice(range(1, n), size=rng.integers(0, n - 1), replace=False)) if n > 2 else []
    blocks = [list(b) for b in np.split(perm, cut)]
    for name, F in (("Shannon", H), ("collision", H2), ("min", Hinf)):
        worst[name] = max(worst[name], abs(grouping_defect(F, list(p), blocks)))
print("largest grouping defect:", {k: round(v, 6) for k, v in worst.items()})
print("H2(1/3,1/3,1/3) = %.3f, grouped: %.3f" % (H2([1/3] * 3), H2([1/3, 2/3]) + 2/3 * H2([0.5, 0.5])))
for _ in range(5000):
    p = rng.dirichlet(np.ones(6) * 0.3)
    assert Hinf(p) <= H2(p) + 1e-9 <= H(p) + 2e-9 and H2(p) <= 2 * Hinf(p) + 1e-9
p = [0.5] + [2.0 ** -21] * 2 ** 20
print("atom 1/2 + 2^20 points: Hinf = %.3f, H2 = %.3f, H = %.3f" % (Hinf(p), H2(p), H(p)))

### P1.7

In [ ]:
from math import comb, sqrt
for k in (4, 10, 100, 1000, 10000):
    m, T = comb(k, 2), comb(k, 3)
    print("K_%d: triangles / bound = %.4f" % (k, T / (sqrt(2) / 3 * m ** 1.5)))
for n, pr in ((60, 0.1), (60, 0.5), (200, 0.05)):
    A = np.triu(rng.random((n, n)) < pr, 1); A = (A | A.T).astype(int)
    T = int(np.trace(A @ A @ A) // 6); m = int(A.sum() // 2)
    print("G(%d,%.2f): m=%d, triangles=%d, bound=%.0f" % (n, pr, m, T, sqrt(2) / 3 * m ** 1.5))

def lw(S):
    S = set(S)
    return len(S) ** 2, len({(x, y) for x, y, z in S}) * len({(y, z) for x, y, z in S}) * len({(x, z) for x, y, z in S})
for size in (10, 100, 500):
    S = {tuple(rng.integers(0, 10, 3)) for _ in range(size)}
    print("random set: |S|^2 = %d <= %d" % lw(S))
print("box 2x3x5: |S|^2 = %d = %d" % lw(itertools.product(range(2), range(3), range(5))))

n = 100
ok = all(2 ** (n * h(k / n)) / (n + 1) <= comb(n, k) <= 2 ** (n * h(k / n)) * (1 + 1e-12) for k in range(n + 1))
ok2 = all(sum(comb(n, j) for j in range(int(lam * n) + 1)) <= 2 ** (n * h(lam)) for lam in np.linspace(0.01, 0.5, 50))
print("binomial bounds hold for n=100:", ok, ok2)
lg = lambda n: math.lgamma(n + 1) / math.log(2)
print("log2 10! = %.2f, log2 1000! = %.1f, Stirling main terms: %.1f" % (lg(10), lg(1000), 1000 * math.log2(1000) - 1000 / math.log(2)))

The triangle bound is reached asymptotically by complete graphs only; random graphs are far below it, and boxes are the equality case of Loomis–Whitney.